# Three-way kernel-latency comparison v2 — QAT vs PTQ vs FP16

Honest, raw analysis of the trtexec `--exportProfile` JSONs. **Every kernel is a real fused
TensorRT kernel** — `QuantizeLinear` appears *folded into* conv names, never as a standalone
'Q/DQ pair'. We use the raw names as-is; the only standalone quantize kernels are the
`__myl_MulMinMax`/`CastMul` activation-quantize ops (categorised 'activation quant').
**Analysis only** — the three JSONs are read-only inputs.


## STEP 1 — load (medianMs throughout; skip the `{count}` header)

In [ ]:
import json, re
import pandas as pd, numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
pd.set_option('display.max_rows', 400); pd.set_option('display.max_colwidth', 90)

KJ = Path('../Per_kernel_json_file')
FILES = {'QAT':'qat_batch32_per_kernel_profile.json',
         'PTQ':'ptq_int8_per_kernel_profile.json',
         'FP16':'fp16_per_kernel_profile.json'}
ANCHOR = {'QAT':1.200, 'PTQ':1.070, 'FP16':1.127}   # measured whole-engine medians (exclusive A100)
COLORS = {'QAT':'#c0392b', 'PTQ':'#2980b9', 'FP16':'#27ae60'}

def load_profile(path):
    raw = json.load(open(path))
    rows = [e for e in raw if 'name' in e]          # element [0] is {'count': N}
    df = pd.DataFrame(rows)[['name','averageMs','medianMs','percentage']]
    for c in ['averageMs','medianMs','percentage']: df[c] = pd.to_numeric(df[c])
    return df

raw_df = {t: load_profile(KJ/FILES[t]) for t in FILES}
for t, df in raw_df.items():
    print(f'{t:5} kernels={len(df):4d}  sum(medianMs)={df["medianMs"].sum():.4f} ms  '
          f'(whole-engine measured {ANCHOR[t]} ms)')


## STEP 2 — parse the RAW name: layer, subpath, op_type
`reformat/copy` is checked first because those names contain 'Conv'. The QuantizeLinear inside
a fused conv kernel is **folded in** — the kernel is 'conv+SiLU fused', its latency is the whole
fused kernel (not a separable Q cost).

In [ ]:
def op_type(n):
    # reformat/copy FIRST (these names contain 'Conv'; ' copy' suffix has a space)
    if 'Reformatting' in n or 'CopyNode' in n or 'copy' in n: return 'reformat/copy'
    has_conv = 'Conv' in n; has_pwn = 'PWN' in n
    if has_conv and has_pwn and ('Sigmoid' in n or 'Mul' in n): return 'conv+SiLU fused'
    if has_conv and not has_pwn: return 'conv only'
    if n.startswith('PWN(') and 'Sigmoid' in n and 'Mul' in n and 'Conv' not in n: return 'standalone SiLU'
    if n.startswith('__myl_') and ('MulMinMax' in n or 'CastMul' in n): return 'activation quant'
    if any(k in n for k in ['MatMul','MaxrSub','DivMul','MoveSlicSlicTran','MulAddAdd','/attn/']): return 'attention'
    if any(k in n for k in ['Topk','ReshReshResh','ForeignNode','Gather']): return 'NMS/decode'
    if n.startswith('__myl_') and ('Resh' in n or 'Move' in n or 'Tran' in n) and 'MulMinMax' not in n: return 'reshape/move'
    if 'MaxPool' in n: return 'maxpool'
    if 'Resize' in n or 'Concat' in n or 'Slice_output' in n or 'Split' in n: return 'resize/concat'
    return 'other'

def parse(name):
    m = re.search(r'model[./](\d+)', name)
    layer = f'model.{m.group(1)}' if m else 'other/glue'
    subpath = ''
    if m:
        after = name[m.end():]
        s = re.search(r'[./]([\w.]+?)(?:/conv|/act|/weight|/Conv|\b)', after)
        subpath = s.group(1) if s else ''
    return layer, subpath, op_type(name)

def build_df(tag):
    df = load_profile(KJ/FILES[tag])
    p = df['name'].apply(lambda x: pd.Series(parse(x), index=['layer','subpath','op_type']))
    return pd.concat([df, p], axis=1)

DFS = {t: build_df(t) for t in FILES}
# sanity: 'other' must be tiny (only genuinely-uncategorisable edge kernels)
for t in FILES: print(t, "'other' kernels:", int((DFS[t]['op_type']=='other').sum()))


## STEP 3 — per-file kernel table (sorted by layer, then medianMs desc) + CSV with TOTAL row

In [ ]:
def per_file_table(tag):
    d = DFS[tag].copy()
    d['name80'] = d['name'].str.slice(0, 80)
    d = d[['layer','subpath','op_type','medianMs','name80']].sort_values(
            ['layer','medianMs'], ascending=[True, False]).reset_index(drop=True)
    total = pd.DataFrame([{'layer':'TOTAL','subpath':'','op_type':f'{len(d)} kernels',
                           'medianMs':d['medianMs'].sum(),'name80':''}])
    out = pd.concat([d, total], ignore_index=True)
    out.to_csv(f'kernels_{tag}.csv', index=False)
    return out

tbl_QAT  = per_file_table('QAT')
tbl_PTQ  = per_file_table('PTQ')
tbl_FP16 = per_file_table('FP16')
print('saved kernels_QAT.csv / kernels_PTQ.csv / kernels_FP16.csv')
tbl_QAT.head(20)   # preview; swap to tbl_PTQ / tbl_FP16


## STEP 4 — common layers (present in ALL 3), summed median ms + deltas

In [ ]:
common = sorted(set.intersection(*[set(DFS[t]['layer']) for t in FILES]))
rows = {t: DFS[t][DFS[t]['layer'].isin(common)].groupby('layer')['medianMs'].sum() for t in FILES}
comp = pd.DataFrame(rows).reindex(common).fillna(0)
comp['QAT-PTQ']  = comp['QAT'] - comp['PTQ']
comp['QAT-FP16'] = comp['QAT'] - comp['FP16']
comp = comp.sort_values('QAT', ascending=False)
total = pd.DataFrame(comp.sum()).T; total.index = ['TOTAL']
comp_out = pd.concat([comp, total])
comp_out.round(4).to_csv('layer_comparison_common.csv')
print(f'{len(common)} common layers'); comp_out.round(4)


## STEP 5 — grouped bar: per-layer summed median latency (common layers)

In [ ]:
lay = comp.index.tolist(); x = np.arange(len(lay)); w = 0.27
fig, ax = plt.subplots(figsize=(13, 4.8))
for i, t in enumerate(['QAT','PTQ','FP16']):
    ax.bar(x+(i-1)*w, comp[t], w, label=t, color=COLORS[t])
ax.set_xticks(x); ax.set_xticklabels(lay, rotation=45, ha='right')
ax.set_ylabel('summed median latency (ms)')
ax.set_title('Per-layer summed median latency — QAT vs PTQ vs FP16 (common layers)')
ax.legend(); plt.tight_layout(); plt.savefig('layer_comparison.png', dpi=130); plt.show()


## STEP 6 — per-engine op-type summary (sum ms + count) + CSV with TOTAL row

In [ ]:
def op_summary(tag):
    d = DFS[tag].groupby('op_type').agg(sum_ms=('medianMs','sum'), n_kernels=('medianMs','size'))
    d = d.sort_values('sum_ms', ascending=False)
    total = pd.DataFrame({'sum_ms':[d['sum_ms'].sum()], 'n_kernels':[int(d['n_kernels'].sum())]}, index=['TOTAL'])
    out = pd.concat([d, total]); out.round(4).to_csv(f'op_summary_{tag}.csv')
    return out

for t in FILES:
    print(f'--- {t} ---'); display(op_summary(t).round(4))


## STEP 7 — op-type histograms (per engine) + combined grouped bar

In [ ]:
op_ms = pd.DataFrame({t: DFS[t].groupby('op_type')['medianMs'].sum() for t in FILES}).fillna(0)
order = op_ms.sum(axis=1).sort_values(ascending=False).index.tolist()
op_ms = op_ms.reindex(order)

# per-engine horizontal bars
for t in FILES:
    s = op_ms[t][op_ms[t] > 0].sort_values()
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(s.index, s.values, color=COLORS[t])
    for i, v in enumerate(s.values): ax.text(v+0.003, i, f'{v:.3f}', va='center', fontsize=8)
    ax.set_xlabel('summed median ms'); ax.set_title(f'{t} — latency by op-type')
    ax.set_xlim(0, op_ms.values.max()*1.15)
    plt.tight_layout(); plt.savefig(f'optype_{t}.png', dpi=130); plt.show()

# combined grouped bar
x = np.arange(len(op_ms)); w = 0.27
fig, ax = plt.subplots(figsize=(13, 5))
for i, t in enumerate(['QAT','PTQ','FP16']):
    ax.bar(x+(i-1)*w, op_ms[t], w, label=t, color=COLORS[t])
ax.set_xticks(x); ax.set_xticklabels(op_ms.index, rotation=40, ha='right')
ax.set_ylabel('summed median ms'); ax.set_title('Latency by op-type — QAT vs PTQ vs FP16')
ax.legend(); plt.tight_layout(); plt.savefig('optype_combined.png', dpi=130); plt.show()


## STEP 8 — sanity: per-kernel median sums are inflated by CUDA-event instrumentation

In [ ]:
recon = pd.DataFrame({
    'kernels':       {t: len(DFS[t]) for t in FILES},
    'sum_medianMs':  {t: DFS[t]['medianMs'].sum() for t in FILES},
    'measured_ms':   ANCHOR,
})
recon['inflation_x'] = (recon['sum_medianMs']/recon['measured_ms']).round(2)
display(recon.round(4))
print('Per-kernel median sums are ~1.4-1.6x the true whole-engine median (CUDA-event overhead).')
print('Use RELATIVE comparisons across kernels/engines; absolute per-kernel ms are diagnostic only.')
